In [1]:
import pandas as pd

file_path = r'C:\Users\sainz\Downloads\final_clean_merged_owid.xlsx'
df = pd.read_excel(file_path)

df.head()

,country_x,year,iso_code,population_x,gdp,biofuel_elec_per_capita,biofuel_electricity,biofuel_share_elec,carbon_intensity_elec,coal_consumption,...,co2_per_capita,years_in_school_men,years_in_school_women,country_y,Access to electricity (% of population),Current account balance (% of GDP),External balance on goods and services (% of GDP),"Total reserves (includes gold, current US$)",Total reserves minus gold (current US$),renewable_adoption_level
0,Afghanistan,1990,AFG,12045622.0,1.306598e+10,NaN,NaN,NaN,NaN,NaN,...,0.2130,2.26,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1991,AFG,12238831.0,1.204736e+10,NaN,NaN,NaN,NaN,NaN,...,0.1880,2.32,0.43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1992,AFG,13278937.0,1.267754e+10,NaN,NaN,NaN,NaN,NaN,...,0.0997,2.38,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1993,AFG,14943125.0,9.834582e+09,NaN,NaN,NaN,NaN,NaN,...,0.0891,2.44,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1994,AFG,16250755.0,7.919857e+09,NaN,NaN,NaN,NaN,NaN,...,0.0800,2.50,0.48,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df_clean = df.copy()

df_clean.rename(columns={
    'country_x': 'country',
    'population_x': 'population',
}, inplace=True)

In [3]:
df_clean.columns.tolist()

['country',
 'year',
 'iso_code',
 'population',
 'gdp',
 'biofuel_elec_per_capita',
 'biofuel_electricity',
 'biofuel_share_elec',
 'carbon_intensity_elec',
 'coal_consumption',
 'coal_elec_per_capita',
 'coal_electricity',
 'coal_prod_change_twh',
 'coal_prod_per_capita',
 'coal_production',
 'coal_share_elec',
 'electricity_demand',
 'electricity_demand_per_capita',
 'electricity_generation',
 'energy_cons_change_pct',
 'energy_cons_change_twh',
 'energy_per_capita',
 'energy_per_gdp',
 'fossil_elec_per_capita',
 'fossil_electricity',
 'fossil_share_elec',
 'gas_consumption',
 'gas_elec_per_capita',
 'gas_electricity',
 'gas_prod_change_twh',
 'gas_prod_per_capita',
 'gas_production',
 'gas_share_elec',
 'greenhouse_gas_emissions',
 'hydro_elec_per_capita',
 'hydro_electricity',
 'hydro_share_elec',
 'low_carbon_elec_per_capita',
 'low_carbon_electricity',
 'low_carbon_share_elec',
 'net_elec_imports',
 'net_elec_imports_share_demand',
 'nuclear_elec_per_capita',
 'nuclear_electrici

In [4]:
#Final View 1

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

alt.renderers.set_embed_options(padding={'bottom': 100})

# Data preparation
df_clean = df.copy()
df_clean.rename(columns={
    'country_x': 'country',
    'population_x': 'population',
}, inplace=True)

# Melt renewable shares into long format
renewable_long = df_clean.melt(
    id_vars=['country','year'],
    value_vars=[
        'solar_share_elec',
        'wind_share_elec',
        'hydro_share_elec',
        'biofuel_share_elec',
        'other_renewables_share_elec'
    ],
    var_name='source',
    value_name='share'
)

# Rename sources
renewable_long['source'] = renewable_long['source'].replace({
    'solar_share_elec': 'Solar',
    'wind_share_elec': 'Wind',
    'hydro_share_elec': 'Hydro',
    'biofuel_share_elec': 'Biofuel',
    'other_renewables_share_elec': 'Other'
})

renewable_long['share'] = renewable_long['share'].fillna(0)
renewable_long['year'] = renewable_long['year'].astype(int)

# Setting colors
source_order = ['Solar','Wind','Hydro','Biofuel','Other']
source_colors = {
    'Solar': '#FFD700',
    'Wind': '#2ca02c',
    'Hydro': '#1f77b4',
    'Biofuel': '#8B4513',
    'Other': '#7f7f7f'
}
adoption_colors = {
    'High': '#08306b',
    'Medium': '#2171b5',
    'Low': '#6baed6'
}

category_selection = alt.selection_point(fields=['renewable_adoption_level'], bind='legend', empty='all')
country_selection = alt.selection_point(fields=['country'], on='click', empty='none')
source_selection = alt.selection_point(fields=['source'], on='click', empty='none')
show_lines = alt.param(name='show_lines', bind=alt.binding_checkbox(name='Show Regression Lines'), value=True)
show_points = alt.param(name='show_points', bind=alt.binding_checkbox(name='Show Scatter Points'), value=True)

# Scatter plot
scatter = alt.Chart(df_clean[df_clean['year'] == 2018]).mark_circle(size=80).encode(
    alt.X('life_expectancy:Q', title='Life Expectancy'),
    alt.Y('income:Q', title='GDP per capita (PPP, annual)'),
    color=alt.condition(
        category_selection | country_selection,
        alt.Color('renewable_adoption_level:N',
                  sort=['High','Medium','Low'],
                  scale=alt.Scale(domain=list(adoption_colors.keys()),
                                  range=list(adoption_colors.values())),
                  legend=alt.Legend(
                      title="Renewable Adoption Level",
                      orient="bottom"
                  )),
        alt.value('lightgrey')
    ),
    opacity=alt.condition(category_selection | country_selection, alt.value(1.0), alt.value(0.25)),
    tooltip=[
    alt.Tooltip('country:N', title='Country'),
    alt.Tooltip('income:Q', title='GDP per capita'),
    alt.Tooltip('life_expectancy:Q', title='Life Expectancy'),
    alt.Tooltip('renewable_adoption_level:N', title='Renewable Adoption Level')
]
).transform_filter(show_points).add_params(category_selection, country_selection, show_lines, show_points
).properties(
    width=400,
    height=300
)

regression = alt.Chart(df_clean[df_clean['year'] == 2018]).transform_regression(
    'life_expectancy','income', groupby=['renewable_adoption_level']
).mark_line(size=3).encode(
    x='life_expectancy:Q',
    y='income:Q',
    color=alt.Color('renewable_adoption_level:N',
                    sort=['High','Medium','Low'],
                    scale=alt.Scale(domain=list(adoption_colors.keys()),
                                    range=list(adoption_colors.values())),
                    legend=None)
).transform_filter(category_selection).transform_filter(show_lines).add_params(category_selection, country_selection, show_lines, show_points)

scatter_regression = (scatter + regression).properties(width=400, height=350, title='GDP per capita vs Life Expectancy').interactive()

# Stacked area chart
country_title = alt.Chart(df_clean[['country']].drop_duplicates()).mark_text(
    align='center', fontSize=16, fontWeight='bold'
).encode(text='country:N').transform_filter(country_selection).properties(width=700, height=30)

stacked_area = alt.Chart(renewable_long).mark_area().encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('share:Q', title='Share of Electricity (%)',
            stack='zero', scale=alt.Scale(domain=[0,100])),
    color=alt.Color('source:N',
                    sort=source_order,
                    scale=alt.Scale(domain=list(source_colors.keys()),
                                    range=list(source_colors.values())),
                    legend=alt.Legend(title='Renewable Source',
                                      orient='bottom',
                                      direction='horizontal')),
    opacity=alt.condition(source_selection, alt.value(1.0), alt.value(0.35)),
    tooltip=[
    alt.Tooltip('country:N', title='Country'),
    alt.Tooltip('year:O', title='Year'),
    alt.Tooltip('source:N', title='Source'),
    alt.Tooltip('share:Q', title='Share')
]
).transform_filter(country_selection).properties(width=600, height=300).add_params(source_selection, country_selection)

stacked_area_with_title = alt.vconcat(country_title, stacked_area, spacing=0)

controls_anchor = alt.Chart(pd.DataFrame({'anchor':[0]})).mark_point().encode(
    x=alt.X('anchor:Q', axis=None)
).properties(width=500, height=5, title='').add_params(show_lines, show_points)

scatter_with_controls = alt.vconcat(scatter_regression, controls_anchor, spacing=0)

# 2024 ranks
renewable_2024 = renewable_long[(renewable_long['year'] == 2024)].copy()
ranked_2024 = (
    renewable_2024
    .sort_values(['source','share'], ascending=[True, False])
    .assign(rank=lambda df: df.groupby('source').cumcount() + 1)
)
ranked_2024['rank'] = ranked_2024['rank'].astype(int)

TOP_N = 10
top10 = ranked_2024[ranked_2024['rank'] <= TOP_N]

# Comparison chart
comparison_base = alt.Chart(top10).transform_filter(
    source_selection
).transform_filter(
    alt.datum.year == 2024
)

bars = comparison_base.mark_bar().encode(
    x=alt.X('share:Q', title='Source Percent (%)'),
    y=alt.Y('country:N', sort='-x'),
    # Highlighting if selected country is in top 10
    color=alt.condition(
        country_selection,
        alt.value('crimson'),
        alt.Color('source:N',
                  sort=source_order,
                  scale=alt.Scale(domain=list(source_colors.keys()),
                                  range=list(source_colors.values())),
                  legend=None)
    ),
    tooltip=[
    alt.Tooltip('country', title='Country'),
    alt.Tooltip('source', title='Source'),
    alt.Tooltip('share', title='Share')
    ]
).properties(
    width=500,
    height=300,
    title='Top 10 Countries by Renewable Electricity Share (2024)'
).add_params(source_selection, country_selection)

comparison_chart = bars

# Combining the graphs
right_side = alt.vconcat(
    stacked_area_with_title,
    comparison_chart,
    spacing=20
)

chart = alt.hconcat(
    scatter_with_controls,
    right_side,
    spacing=30
).resolve_scale(color='independent')

# General dashboard title
dashboard_title = alt.Chart(pd.DataFrame({'text': ['Global Renewable Energy Adoption and Socioeconomic Development']})).mark_text(
    align='center',
    fontSize=22,
    fontWeight='bold'
).encode(
    text='text'
).properties(width=1000, height=40)

dashboard = alt.vconcat(
    dashboard_title,
    chart, 
    spacing=10
)

dashboard

alt.VConcatChart(...)